In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel
import torch.nn as nn
from torch.optim import Adam
from tqdm import tqdm


In [ ]:
# Load the dataset
file_path = 'DVD11.csv'  # Replace with your training dataset path
test_file_path = 'Books11.csv'  # Replace with your test dataset path

train_data = pd.read_csv(file_path)
test_data = pd.read_csv(test_file_path)

train_data['sentiment'] = train_data['star_rating']
test_data['sentiment'] = test_data['star_rating']

# Encode sentiments
le = LabelEncoder()
train_data['sentiment_encoded'] = le.fit_transform(train_data['sentiment'])
test_data['sentiment_encoded'] = le.transform(test_data['sentiment'])

# Tokenize text using BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_data(texts, tokenizer, max_length):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

MAX_LENGTH = 128
train_encodings = tokenize_data(train_data['review_body'], tokenizer, MAX_LENGTH)
test_encodings = tokenize_data(test_data['review_body'], tokenizer, MAX_LENGTH)

train_labels = torch.tensor(train_data['sentiment_encoded'].values)
test_labels = torch.tensor(test_data['sentiment_encoded'].values)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = SentimentDataset(train_encodings, train_labels)
test_dataset = SentimentDataset(test_encodings, test_labels)


In [ ]:
class BertBiLSTMClassifier(nn.Module):
    def __init__(self, n_classes):
        super(BertBiLSTMClassifier, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.lstm = nn.LSTM(input_size=768, hidden_size=128, num_layers=1, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(128 * 2, n_classes)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        lstm_out, _ = self.lstm(outputs.last_hidden_state)
        avg_pool = torch.mean(lstm_out, 1)
        logits = self.fc(avg_pool)
        return logits


In [ ]:
def train_model(model, train_loader, val_loader, optimizer, criterion, device, epochs=3):
    train_loss_history = []
    val_loss_history = []
    train_acc_history = []
    val_acc_history = []

    for epoch in range(epochs):
        # Training phase
        model.train()
        total_loss = 0
        correct_train_preds = 0
        total_train_samples = 0

        for batch in tqdm(train_loader, desc=f"Training Epoch {epoch + 1}"):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Backward pass and optimizer step
            loss.backward()
            optimizer.step()

            # Calculate training accuracy
            preds = torch.argmax(outputs, dim=1)
            correct_train_preds += (preds == labels).sum().item()
            total_train_samples += labels.size(0)

        # Compute average training loss and accuracy
        avg_train_loss = total_loss / len(train_loader)
        train_accuracy = correct_train_preds / total_train_samples

        # Validation phase
        model.eval()
        val_loss = 0
        correct_val_preds = 0
        total_val_samples = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Validation Epoch {epoch + 1}"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                # Forward pass
                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                # Calculate validation accuracy
                preds = torch.argmax(outputs, dim=1)
                correct_val_preds += (preds == labels).sum().item()
                total_val_samples += labels.size(0)

        # Compute average validation loss and accuracy
        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = correct_val_preds / total_val_samples

        # Store metrics for plotting
        train_loss_history.append(avg_train_loss)
        train_acc_history.append(train_accuracy)
        val_loss_history.append(avg_val_loss)
        val_acc_history.append(val_accuracy)

        # Print results for the epoch
        print(f"\nEpoch {epoch + 1}/{epochs}")
        print(f"Training Loss: {avg_train_loss:.3f}, Training Accuracy: {train_accuracy:.3f}")
        print(f"Validation Loss: {avg_val_loss:.3f}, Validation Accuracy: {val_accuracy:.3f}")

    return train_loss_history, train_acc_history, val_loss_history, val_acc_history


In [ ]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(dataset, test_size=0.2, random_state=42)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

train_loss, train_acc, val_loss, val_acc = train_model(
    model, train_loader, val_loader, optimizer, criterion, DEVICE, epochs=EPOCHS
)


In [ ]:
import matplotlib.pyplot as plt

# Plot Training vs Validation Accuracy
plt.figure(figsize=(10, 5))
plt.plot(train_acc, label="Training Accuracy")
plt.plot(val_acc, label="Validation Accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# Plot Training vs Validation Loss
plt.figure(figsize=(10, 5))
plt.plot(train_loss, label="Training Loss")
plt.plot(val_loss, label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
def evaluate_model(model, test_loader, device, label_encoder):
    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)
            _, preds = torch.max(outputs, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    accuracy = accuracy_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions, average='weighted')
    precision = precision_score(true_labels, predictions, average='weighted')
    recall = recall_score(true_labels, predictions, average='weighted')

    # Print metrics
    print("Evaluation Metrics:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")

    # Decode integer labels back to string labels for classification report
    target_names = label_encoder.inverse_transform(range(len(label_encoder.classes_)))

    # Classification Report
    print("\nClassification Report:")
    print(classification_report(true_labels, predictions, target_names=target_names))


In [ ]:
# Hyperparameters
BATCH_SIZE = 16
EPOCHS = 40
LEARNING_RATE = 1e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Initialize model, optimizer, and loss function
model = BertBiLSTMClassifier(n_classes=len(le.classes_)).to(DEVICE)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

# Train the model
train_model(model, train_loader, optimizer, criterion, DEVICE, epochs=EPOCHS)




100%|██████████| 778/778 [01:32<00:00,  8.40it/s]


Epoch 1/40
Training Loss: 0.388, Training Accuracy: 0.860


100%|██████████| 778/778 [01:32<00:00,  8.44it/s]


Epoch 2/40
Training Loss: 0.262, Training Accuracy: 0.889


100%|██████████| 778/778 [01:31<00:00,  8.47it/s]


...

100%|██████████| 778/778 [01:32<00:00,  8.44it/s]


Epoch 40/40
Training Loss: 0.122, Training Accuracy: 0.953


Evaluating: 100%|██████████| 2070/2070 [03:55<00:00,  8.78it/s]


Evaluation Metrics:
Accuracy: 0.8835
F1 Score: 0.8781
Precision: 0.8745
Recall: 0.8835

Classification Report:


TypeError: object of type 'numpy.int64' has no len()

In [ ]:
# Evaluate the model
print(le.classes_)  # Should print: ['Negative', 'Positive']
from sklearn.preprocessing import LabelEncoder

# Initialize and fit the LabelEncoder
le = LabelEncoder()
le.fit(["Negative", "Positive"])  # Ensure these are your class names


evaluate_model(model, test_loader, DEVICE, le)

[0 1]


Evaluating: 100%|██████████| 2070/2070 [03:56<00:00,  8.77it/s]


Evaluation Metrics:
Accuracy: 0.8835
F1 Score: 0.8781
Precision: 0.8745
Recall: 0.8835

Classification Report:
              precision    recall  f1-score   support

    Negative       0.56      0.45      0.50      4253
    Positive       0.92      0.95      0.93     28860

    accuracy                           0.88     33113
   macro avg       0.74      0.70      0.72     33113
weighted avg       0.87      0.88      0.88     33113

